# Morse Potential

$$
V(x) = D_e \left(1 - e^{-a x}\right)^2
$$

where
$$
E_n = \hbar a \sqrt{\frac{2 D_e}{m}} \left(n + \tfrac{1}{2}\right)
- \frac{\hbar^2 a^2}{2m} \left(n + \tfrac{1}{2}\right)^2
$$


By substituting $\hbar=m=1$, $D_e = 8, a=0.5$ the expected results are
```
Expected : [0.96875, 2.71875, 4.21875,...]
```
with respect to `n = [0, 1, 2,...]`

### Setup

In [5]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

m, hbar = 1.0, 1.0

def V(x):
    D_e = 8.0
    a = 0.5
    return D_e * (1.0 - np.exp(-a * x))**2

def func(x, u_vec, E):
    u, du = u_vec
    d2u = (2.0 * m / hbar**2) * (V(x) - E) * u
    return [du, d2u]

def shoot(E):
    sol = solve_ivp(fun=func, t_span=(-L, L), y0=[0, 1], method="RK45", args=(E,), atol=1e-8, rtol=1e-8)

    # u_final = u(L)
    u_final = sol.y[0, -1]

    # if |u_final|>1e10, return u_final to np.sign(u_final) * 1e10 to avoid overflow
    return np.sign(u_final) * min(abs(u_final), 1e10)

### Find optimal L
We guess
```
E_min, E_max = -10, 10
```
We defined `find_optimal()` and find $L_{optimal}$ with

```
for L in np.arange(2, 20, 2):
...
```

In [6]:
E_min, E_max = -5, 5

# Defind find_optimalL
def find_optimalL(E_min, E_max, tol=1e-6):
    print(f"{'L':<10} | {'E_root'}")
    print("-" * 25)
    E_prev = 0.0

#  Change L_min, L_max, and nsteps at here eg. Lmin = 2, Lmax=10, nsteps = 1
    for L in np.arange(2, 20, 2):

        def shoot(E):
            sol = solve_ivp(fun=func, t_span=(-L, L), y0=[0, 1], method="RK45", args=(E,), atol=1e-8, rtol=1e-8)

            u_final = sol.y[0, -1]

            return np.sign(u_final) * min(abs(u_final), 1e10)

        energies = np.linspace(E_min, E_max, 100)
        eigenvalues=[]
        f_old = shoot(energies[0])

        for i in range(len(energies) - 1):
            E_low = energies[i]
            E_high = energies[i+1]
            f_new = shoot(E_high)

            if f_old * f_new < 0:
                try:
                    E_root = brentq(shoot, E_low, E_high)
                    eigenvalues.append(E_root)
                    break
                except ValueError:
                    pass

            f_old = f_new

        if not eigenvalues:
            print(f"{L:<10.1f} | No eigenvalues found")
            continue

        E_root = eigenvalues[0]
        print(f"{L:<10.1f} | {E_root:.8f}")

        if abs(E_root - E_prev) < tol:
            print("-" * 25)
            print(f"Stability reached at L = {L:.1f}")

            return L, E_root
        E_prev = E_root

# Execute to find optimal L
optimal_L, stable_E = find_optimalL(E_min, E_max, tol=1e-6)

L          | E_root
-------------------------
2.0        | 0.98053563
4.0        | 0.96875022
6.0        | 0.96875000
-------------------------
Stability reached at L = 6.0


> Output : `L = 5.0 and E0 > 0.9`

Thence, we define and run `find_eigenvalues()` with L = 6

We modify `E_min, E_max = 0.9, 5` to speed up

In [7]:
L = 6.0
E_min, E_max = 0.9, 5
# Define find_eigenvalues()
def find_eigenvalues(E_min, E_max, n, shoot=shoot):
    energies = np.linspace(E_min, E_max, n)
    eigenvalues = []

    f_old = shoot(energies[0])

    for i in range(len(energies) - 1):
        E_low = energies[i]
        E_high = energies[i+1]
        f_new = shoot(E_high)

        '''Sign change detected -> root found in this interval'''
        if f_old * f_new < 0:
            E_root = brentq(shoot, E_low, E_high)
            eigenvalues.append(E_root)

        f_old = f_new

    print(eigenvalues)

    if not eigenvalues:
        print("No eigenvalues found. Try different L or E_min, E_max.")

# Execute find_eigenvalues()
find_eigenvalues(E_min, E_max, n=100)

[0.9687499998338143, 2.7187500004342935, 4.21875073710904]


When `L = 6.0` the eigenvalues give
```
[0.9687499998338143, 2.7187500004342935, 4.21875073710904]
```
Highly aligns what we expected : [0.96875, 2.71875, 4.21875,...]
